# Indian Sign Language (ISL) Recognition
### DataMites Capstone Project — PRAICP-1000-IndiSignLang

**Objective:** Build an image-classification system that predicts the sign class represented by a hand-gesture image.

**Project brief alignment:** The supplied project brief identifies two core goals: image preprocessing and an ML model to predict the sign. It describes the dataset as directories containing sign images.

> **Important:** This notebook does not invent evaluation scores. All model results, charts, and conclusions are generated from the dataset when you run the notebook from top to bottom.

**Notebook design principles**
- One consistent dataset and label mapping throughout the notebook.
- Stratified train/validation/test partitions where class counts permit.
- Preprocessing is learned/fitted only from training data.
- Four classical ML baselines use HOG image features; one CNN learns image features directly.
- Validation data is used for model selection; the held-out test set is used for the final evaluation.
- Metrics include accuracy, macro precision/recall/F1, weighted F1, and confusion matrix.

## 0. How to prepare the dataset

1. Download the ZIP dataset from the project brief:  
   https://d3ilbtxij3aepc.cloudfront.net/projects/AI-Capstone-Projects/PRAICP-1000-IndiSignLang.zip
2. Extract it on your computer.
3. Set `DATASET_LOCATION` below to the extracted folder. The folder may contain another nested folder; the discovery code searches a few levels for class folders.
4. A typical image-folder layout looks like this:

```text
dataset/
    Class_A/
        image001.jpg
        image002.jpg
    Class_B/
        image001.jpg
        image002.jpg
    Class_C/
        ...
```

Each class folder name becomes the target label. If your dataset uses a different layout (for example, separate train/test folders or a CSV label file), inspect the directory output and adjust the path/layout before continuing.

## 1. Install and import libraries

**Purpose:** Install/import tools for image handling, numerical operations, classical ML, evaluation, visualization, and the CNN.

Run the installation cell only if a package is missing. Restart the Jupyter kernel if TensorFlow was newly installed.

In [ ]:
# If required, uncomment and run this installation command:
# %pip install -U numpy pandas matplotlib seaborn pillow scikit-learn scikit-image joblib tensorflow

import os
import sys
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from skimage.feature import hog
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Core libraries imported successfully.")

## 2. Project configuration

**Purpose:** Keep image size, random seed, and dataset path in one place. This avoids inconsistent settings across cells.

Change only `DATASET_LOCATION` to the folder where you extracted the project ZIP. Use a raw string (`r"..."`) for Windows paths to avoid backslash errors.

In [ ]:
# Example Windows path:
DATASET_LOCATION = r"C:\Users\YourName\Downloads\PRAICP-1000-IndiSignLang"

# Example if the extracted folder is inside the current notebook directory:
# DATASET_LOCATION = "PRAICP-1000-IndiSignLang"

IMAGE_SIZE = (96, 96)       # height, width used consistently by all models
TEST_SIZE = 0.15            # final untouched test partition
VALIDATION_SIZE = 0.15      # validation partition used for model selection
BATCH_SIZE = 32
CNN_EPOCHS = 20

dataset_path = Path(DATASET_LOCATION)
print("Dataset path exists:", dataset_path.exists())
print("Configured path:", dataset_path.resolve() if dataset_path.exists() else dataset_path)

## 3. Locate class folders and inventory the dataset

**Purpose:** The project brief says images are organized in directories, but it does not specify the exact extracted folder nesting. This cell searches a maximum of three directory levels and selects the folder whose immediate subfolders contain image files.

**Validation:** The selected directory must contain at least two class folders. Review the printed path and class counts before proceeding. Do not continue if the wrong folder is selected.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    return sum(1 for p in folder.rglob("*")
               if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def find_class_root(start_path, max_depth=3):
    if not start_path.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {start_path}. Update DATASET_LOCATION in Section 2."
        )
    candidates = [start_path]
    for current, dirs, files in os.walk(start_path):
        current_path = Path(current)
        depth = len(current_path.relative_to(start_path).parts)
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if depth < max_depth:
            candidates.extend(current_path / d for d in dirs)

    valid = []
    for candidate in candidates:
        class_dirs = [d for d in candidate.iterdir()
                      if d.is_dir() and not d.name.startswith(".")]
        usable = [(d, count_images(d)) for d in class_dirs]
        usable = [(d, n) for d, n in usable if n > 0]
        if len(usable) >= 2:
            valid.append((sum(n for _, n in usable), candidate, usable))

    if not valid:
        raise ValueError(
            "Could not find at least two class folders containing images. "
            "Check the extracted dataset layout and DATASET_LOCATION."
        )
    valid.sort(key=lambda x: x[0], reverse=True)
    return valid[0][1], valid[0][2]

CLASS_ROOT, class_inventory = find_class_root(dataset_path)
print("Detected class root:", CLASS_ROOT)
inventory_df = pd.DataFrame(
    [{"class_name": folder.name, "image_count": n} for folder, n in class_inventory]
).sort_values("class_name").reset_index(drop=True)
display(inventory_df)
print("Number of classes:", len(inventory_df))
print("Total images in detected classes:", int(inventory_df["image_count"].sum()))

## 4. Create a clean image manifest

**Purpose:** Build one table containing each image path and its class label. This manifest is the single source of truth for splitting, training, and evaluation.

Unreadable images are skipped and counted. Duplicate file paths are removed. Class labels are derived from folder names, not guessed from image content.

In [ ]:
records = []
bad_files = []

for class_dir in sorted([d for d in CLASS_ROOT.iterdir() if d.is_dir()]):
    for image_path in class_dir.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                with Image.open(image_path) as img:
                    img.verify()
                records.append({"image_path": str(image_path), "label": class_dir.name})
            except Exception:
                bad_files.append(str(image_path))

image_df = pd.DataFrame(records).drop_duplicates(subset=["image_path"]).reset_index(drop=True)
if image_df.empty:
    raise ValueError("No readable images were found. Check the dataset directory.")

print("Readable images:", len(image_df))
print("Unreadable images skipped:", len(bad_files))
print("Manifest dimensions:", image_df.shape)
display(image_df.head())
print("\nImages per class:")
display(image_df["label"].value_counts().sort_index().rename("count").to_frame())

## 5. Exploratory data analysis (EDA)

**Purpose:** Understand class balance and inspect sample images before training. This can reveal missing classes, severe imbalance, incorrect folder labels, or poor-quality images.

The bar chart shows the number of images in each class. The image grid displays example images with their folder-derived labels.

In [ ]:
class_counts = image_df["label"].value_counts().sort_index()
plt.figure(figsize=(max(10, len(class_counts) * 0.55), 5))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Image Count by Sign Class")
plt.xlabel("Sign class")
plt.ylabel("Number of images")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

sample_n = min(12, len(image_df))
sample_df = image_df.sample(sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax in axes.flat:
    ax.axis("off")
for ax, row in zip(axes.flat, sample_df.itertuples(index=False)):
    with Image.open(row.image_path) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(str(row.label))
    ax.axis("off")
plt.suptitle("Random Sample of ISL Images")
plt.tight_layout()
plt.show()

## 6. Data quality checks and label encoding

**Purpose:** Confirm there are enough examples per class for stratified splitting. Encode text class names into stable integer IDs; the mapping is saved and reused for every model.

A class with fewer than three readable images cannot reliably appear in train, validation, and test partitions. This notebook stops with a clear message rather than silently creating invalid splits.

In [ ]:
label_names = sorted(image_df["label"].unique().tolist())
label_to_id = {name: idx for idx, name in enumerate(label_names)}
id_to_label = {idx: name for name, idx in label_to_id.items()}
image_df["label_id"] = image_df["label"].map(label_to_id).astype(int)

small_classes = image_df["label"].value_counts()
small_classes = small_classes[small_classes < 3]
if len(small_classes):
    raise ValueError(
        "At least one class has fewer than 3 images, so a stratified train/validation/test "
        f"split is not reliable. Classes: {small_classes.to_dict()}"
    )

print("Number of classes:", len(label_names))
print("Label mapping:")
display(pd.DataFrame({"label_id": list(id_to_label.keys()),
                      "class_name": list(id_to_label.values())}))

## 7. Stratified train, validation, and test split

**Purpose:** Split image paths before fitting any model. Stratification preserves class proportions across partitions.

- **Training set:** used to learn model parameters.
- **Validation set:** used to compare/select models and tune settings.
- **Test set:** held aside until the final evaluation.

The test set is not used for model selection. If a class has too few examples for this split, the earlier data-quality check prevents a misleading evaluation.

In [ ]:
train_val_df, test_df = train_test_split(
    image_df, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=image_df["label_id"]
)
relative_val = VALIDATION_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df, test_size=relative_val, random_state=RANDOM_STATE,
    stratify=train_val_df["label_id"]
)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} images | {part['label'].nunique()} classes")
    if set(part["label_id"].unique()) != set(image_df["label_id"].unique()):
        raise ValueError(f"{name} partition is missing one or more classes.")

split_summary = pd.DataFrame({
    "partition": ["Train", "Validation", "Test"],
    "images": [len(train_df), len(val_df), len(test_df)],
    "percentage": [len(train_df)/len(image_df)*100,
                   len(val_df)/len(image_df)*100,
                   len(test_df)/len(image_df)*100]
})
display(split_summary.round(2))

## 8. Image preprocessing

**Purpose:** Convert each image to RGB, resize it to the same dimensions, and scale pixel values to the range 0–1 for the CNN.

Resizing makes image tensors consistent. RGB conversion handles grayscale/RGBA files consistently. The same deterministic resize/scale operation is applied to all partitions; no statistics are learned from validation or test images.

In [ ]:
def load_rgb_image(path, size=IMAGE_SIZE):
    with Image.open(path) as img:
        img = img.convert("RGB")
        img = img.resize((size[1], size[0]), Image.Resampling.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

def load_partition(frame):
    images, labels = [], []
    for row in frame.itertuples(index=False):
        images.append(load_rgb_image(row.image_path))
        labels.append(int(row.label_id))
    return np.stack(images), np.asarray(labels, dtype=np.int64)

X_train_img, y_train = load_partition(train_df)
X_val_img, y_val = load_partition(val_df)
X_test_img, y_test = load_partition(test_df)

print("Train image tensor:", X_train_img.shape, X_train_img.dtype)
print("Validation image tensor:", X_val_img.shape)
print("Test image tensor:", X_test_img.shape)
print("Pixel range:", float(X_train_img.min()), "to", float(X_train_img.max()))

## 9. Feature extraction for classical ML models (HOG)

**Concept:** Classical ML algorithms generally need a numeric feature vector rather than a raw image. Histogram of Oriented Gradients (HOG) summarizes local edge directions and shapes, which can represent hand contours and finger arrangements.

HOG is used here as a baseline feature extractor. It is not guaranteed to outperform a CNN; the validation results will show how it performs on this dataset.

In [ ]:
def extract_hog_features(images):
    features = []
    for image in images:
        gray = np.dot(image, [0.299, 0.587, 0.114])
        feat = hog(
            gray, orientations=9, pixels_per_cell=(8, 8),
            cells_per_block=(2, 2), block_norm="L2-Hys",
            feature_vector=True
        )
        features.append(feat)
    return np.asarray(features, dtype=np.float32)

X_train_hog = extract_hog_features(X_train_img)
X_val_hog = extract_hog_features(X_val_img)
X_test_hog = extract_hog_features(X_test_img)

print("HOG train feature matrix:", X_train_hog.shape)
print("HOG validation feature matrix:", X_val_hog.shape)
print("HOG test feature matrix:", X_test_hog.shape)

## 10. Define common evaluation functions

**Purpose:** Use the same metrics for every model so comparisons are fair.

- **Accuracy:** fraction of all predictions that are correct.
- **Macro precision/recall/F1:** calculate each class metric separately and average classes equally; useful when classes have different sample counts.
- **Weighted F1:** averages class F1 scores weighted by class support.
- **Confusion matrix:** shows which true classes are confused with which predicted classes.

For a multi-class sign-recognition task, macro F1 and per-class recall are important complements to accuracy.

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

validation_results = {}
validation_predictions = {}
trained_classical_models = {}

## 11. Model 1 — Support Vector Machine (SVM)

**Concept:** SVM finds a decision boundary that separates classes. It is a strong baseline for high-dimensional HOG features. `StandardScaler` is placed inside a pipeline so scaling is fitted only on the training features.

`class_weight="balanced"` adjusts the penalty for classes with fewer training examples.

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf", C=3, gamma="scale",
                        class_weight="balanced", random_state=RANDOM_STATE))
])
svm_model.fit(X_train_hog, y_train)
svm_val_pred = svm_model.predict(X_val_hog)
validation_results["SVM (HOG)"] = calculate_metrics(y_val, svm_val_pred)
validation_predictions["SVM (HOG)"] = svm_val_pred
trained_classical_models["SVM (HOG)"] = svm_model
print("SVM validation results:", validation_results["SVM (HOG)"])

## 12. Model 2 — Logistic Regression

**Concept:** Logistic Regression estimates class probabilities using a linear decision function. It is a useful, interpretable baseline for multi-class classification.

Scaling is performed inside the pipeline and class weighting helps reduce the effect of unequal class frequencies.

In [ ]:
logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000, class_weight="balanced",
        solver="lbfgs", random_state=RANDOM_STATE
    ))
])
logreg_model.fit(X_train_hog, y_train)
logreg_val_pred = logreg_model.predict(X_val_hog)
validation_results["Logistic Regression (HOG)"] = calculate_metrics(y_val, logreg_val_pred)
validation_predictions["Logistic Regression (HOG)"] = logreg_val_pred
trained_classical_models["Logistic Regression (HOG)"] = logreg_model
print("Logistic Regression validation results:", validation_results["Logistic Regression (HOG)"])

## 13. Model 3 — Random Forest

**Concept:** Random Forest combines many decision trees trained on sampled data/features. It can model non-linear relationships and does not require feature scaling.

`class_weight="balanced_subsample"` adjusts class weights within each tree's bootstrap sample.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300, class_weight="balanced_subsample",
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train_hog, y_train)
rf_val_pred = rf_model.predict(X_val_hog)
validation_results["Random Forest (HOG)"] = calculate_metrics(y_val, rf_val_pred)
validation_predictions["Random Forest (HOG)"] = rf_val_pred
trained_classical_models["Random Forest (HOG)"] = rf_model
print("Random Forest validation results:", validation_results["Random Forest (HOG)"])

## 14. Model 4 — K-Nearest Neighbors (KNN)

**Concept:** KNN predicts a class using the labels of nearby training examples in feature space. It is a useful distance-based baseline. Scaling is essential because distance calculations are sensitive to feature magnitudes.

The chosen `n_neighbors` is a baseline setting; it can be tuned using training-only cross-validation later.

In [ ]:
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5, weights="distance"))
])
knn_model.fit(X_train_hog, y_train)
knn_val_pred = knn_model.predict(X_val_hog)
validation_results["KNN (HOG)"] = calculate_metrics(y_val, knn_val_pred)
validation_predictions["KNN (HOG)"] = knn_val_pred
trained_classical_models["KNN (HOG)"] = knn_model
print("KNN validation results:", validation_results["KNN (HOG)"])

## 15. Model 5 — Convolutional Neural Network (CNN)

**Concept:** A CNN learns visual features directly from image pixels. Convolution layers learn patterns such as edges and finger shapes; pooling reduces spatial dimensions; dense layers perform classification.

This model uses light data augmentation on training images only. Early stopping monitors validation loss and restores the best validation weights. Class weights are calculated from the training partition only.

**Environment note:** TensorFlow may require installation and a compatible Python version. If import fails, use a supported Python environment (commonly Python 3.10–3.12, depending on the TensorFlow release) and install TensorFlow there.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TensorFlow:", tf.__version__)
tf.keras.utils.set_random_seed(RANDOM_STATE)

num_classes = len(label_names)
class_ids = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=class_ids, y=y_train)
class_weight_dict = {int(k): float(v) for k, v in zip(class_ids, weights)}

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05)
], name="training_augmentation")

cnn_model = models.Sequential([
    layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
    data_augmentation,
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.35),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(num_classes, activation="softmax")
], name="ISL_CNN")

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_model.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
)

history = cnn_model.fit(
    X_train_img, y_train,
    validation_data=(X_val_img, y_val),
    epochs=CNN_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

cnn_val_prob = cnn_model.predict(X_val_img, verbose=0)
cnn_val_pred = np.argmax(cnn_val_prob, axis=1)
validation_results["CNN"] = calculate_metrics(y_val, cnn_val_pred)
validation_predictions["CNN"] = cnn_val_pred
print("CNN validation results:", validation_results["CNN"])

## 16. Inspect CNN learning curves

**Purpose:** Compare training and validation accuracy/loss across epochs. A large gap between training and validation performance may indicate overfitting; poor performance on both may indicate underfitting or insufficient training.

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["accuracy"], label="Training accuracy")
axes[0].plot(history_df["val_accuracy"], label="Validation accuracy")
axes[0].set_title("CNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history_df["loss"], label="Training loss")
axes[1].plot(history_df["val_loss"], label="Validation loss")
axes[1].set_title("CNN Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## 17. Compare the five models on validation data

**Purpose:** Compare all five approaches on the same validation images. The validation set is used to choose the candidate model; the final test set remains untouched until the next stage.

The primary selection metric is **macro F1**, because every sign class receives equal importance in the average. Accuracy and weighted F1 are shown as supporting metrics.

In [ ]:
validation_table = pd.DataFrame(validation_results).T
validation_table = validation_table.sort_values("macro_f1", ascending=False)
display(validation_table.style.format("{:.4f}"))
best_model_name = validation_table.index[0]
print("Selected candidate using validation macro F1:", best_model_name)

ax = validation_table[["accuracy", "macro_f1", "weighted_f1"]].plot(
    kind="bar", figsize=(12, 5), ylim=(0, 1)
)
ax.set_title("Validation Performance — Five Models")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 18. Hyperparameter tuning — SVM baseline

**Purpose:** Demonstrate a leakage-safe tuning workflow. GridSearchCV evaluates parameter combinations using stratified folds from the training partition only. The validation and test partitions are not used to fit the scaler or select parameters within cross-validation.

SVM is tuned as a focused example. The tuned candidate is evaluated on validation data and replaces the original SVM candidate only if macro F1 improves. This keeps the model-selection rule explicit.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
svm_pipeline_for_tuning = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(class_weight="balanced", random_state=RANDOM_STATE))
])
param_grid = {
    "classifier__kernel": ["rbf", "linear"],
    "classifier__C": [0.5, 1, 3, 10],
    "classifier__gamma": ["scale", "auto"]
}
svm_search = GridSearchCV(
    svm_pipeline_for_tuning, param_grid,
    scoring="f1_macro", cv=cv, n_jobs=-1, verbose=1
)
svm_search.fit(X_train_hog, y_train)
tuned_svm = svm_search.best_estimator_
tuned_svm_val_pred = tuned_svm.predict(X_val_hog)
tuned_svm_val_metrics = calculate_metrics(y_val, tuned_svm_val_pred)

print("Best SVM parameters:", svm_search.best_params_)
print("Best training CV macro F1:", round(svm_search.best_score_, 4))
print("Tuned SVM validation metrics:", tuned_svm_val_metrics)

if tuned_svm_val_metrics["macro_f1"] > validation_results["SVM (HOG)"]["macro_f1"]:
    validation_results["Tuned SVM (HOG)"] = tuned_svm_val_metrics
    validation_predictions["Tuned SVM (HOG)"] = tuned_svm_val_pred
    trained_classical_models["Tuned SVM (HOG)"] = tuned_svm
    print("Tuned SVM improves validation macro F1 and is included as a candidate.")
else:
    print("Original SVM remains the candidate; tuning did not improve validation macro F1.")

## 19. Select the final candidate and evaluate once on the test set

**Purpose:** Choose the best candidate using validation macro F1, then evaluate that selected candidate on the held-out test set. The test set is not used to tune or choose the model.

This is the most important unbiased estimate of how the selected model performs on unseen images. Do not change model settings based on the test results and then report the same test score as an unbiased final estimate.

In [ ]:
final_candidate_name = max(validation_results, key=lambda name: validation_results[name]["macro_f1"])
print("Final candidate selected from validation:", final_candidate_name)

if final_candidate_name == "CNN":
    final_test_prob = cnn_model.predict(X_test_img, verbose=0)
    final_test_pred = np.argmax(final_test_prob, axis=1)
    final_model = cnn_model
    final_model_type = "keras"
else:
    final_model = trained_classical_models[final_candidate_name]
    final_test_pred = final_model.predict(X_test_hog)
    final_model_type = "sklearn"

test_metrics = calculate_metrics(y_test, final_test_pred)
print("Held-out test metrics:")
for metric, value in test_metrics.items():
    print(f"{metric:>16}: {value:.4f}")

## 20. Final classification report and confusion matrix

**Purpose:** The classification report gives precision, recall, F1-score, and support for each sign class. The confusion matrix reveals which signs the selected model confuses.

Review low-recall classes carefully. These may need more training images, better image quality, or a different model/augmentation strategy.

In [ ]:
print(classification_report(
    y_test, final_test_pred,
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0
))

cm = confusion_matrix(y_test, final_test_pred, labels=list(range(len(label_names))))
fig_size = max(8, min(18, len(label_names) * 0.55))
plt.figure(figsize=(fig_size, fig_size * 0.85))
sns.heatmap(cm, cmap="Blues", annot=(len(label_names) <= 15), fmt="d",
            xticklabels=label_names, yticklabels=label_names)
plt.title(f"Test Confusion Matrix — {final_candidate_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=75, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 21. Save the final model and label mapping

**Purpose:** Save the selected model and its label mapping so future predictions use the exact same class-to-number relationship.

- CNN is saved in Keras `.keras` format.
- Classical ML model is saved with Joblib.
- Label mapping and preprocessing metadata are saved as JSON.

**Compatibility note:** Scikit-learn Joblib files should be loaded with compatible versions of scikit-learn and its dependencies. To avoid the version mismatch issue experienced in the earlier project, the next cell records the environment versions beside the model.

In [ ]:
import json
import sklearn
import skimage
import PIL

ARTIFACT_DIR = Path("isl_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

if final_model_type == "keras":
    model_file = ARTIFACT_DIR / "isl_cnn_model.keras"
    final_model.save(model_file)
else:
    model_file = ARTIFACT_DIR / "isl_classical_model.joblib"
    joblib.dump(final_model, model_file)

metadata = {
    "selected_model": final_candidate_name,
    "model_type": final_model_type,
    "label_to_id": label_to_id,
    "image_size": list(IMAGE_SIZE),
    "feature_extraction": "HOG" if final_model_type == "sklearn" else "RGB pixels",
    "random_state": RANDOM_STATE,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "scikit_image_version": skimage.__version__,
    "pillow_version": PIL.__version__,
    "tensorflow_version": tf.__version__ if final_model_type == "keras" else None
}
with open(ARTIFACT_DIR / "isl_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved model:", model_file.resolve())
print("Saved metadata:", (ARTIFACT_DIR / "isl_metadata.json").resolve())

## 22. Predict a new sign image

**Purpose:** Test the saved final model on a new image. Set `NEW_IMAGE_PATH` to a separate image file that was not used in training/validation/testing.

For a classical model, the image must be converted to HOG features. For a CNN, the resized RGB image is passed directly to the network. The prediction is translated back to the original class-folder name using the saved label mapping.

In [ ]:
NEW_IMAGE_PATH = r"C:\Users\YourName\Downloads\sample_sign.jpg"

def predict_sign(image_path):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Set NEW_IMAGE_PATH to an existing image: {image_path}")

    image_array = load_rgb_image(image_path)
    if final_model_type == "keras":
        probabilities = final_model.predict(np.expand_dims(image_array, axis=0), verbose=0)[0]
        predicted_id = int(np.argmax(probabilities))
        confidence = float(np.max(probabilities))
    else:
        gray = np.dot(image_array, [0.299, 0.587, 0.114])
        feature = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                      cells_per_block=(2, 2), block_norm="L2-Hys",
                      feature_vector=True).reshape(1, -1)
        predicted_id = int(final_model.predict(feature)[0])
        if hasattr(final_model, "predict_proba"):
            confidence = float(np.max(final_model.predict_proba(feature)))
        elif hasattr(final_model, "decision_function"):
            confidence = None  # decision scores are not calibrated probabilities
        else:
            confidence = None

    predicted_label = id_to_label[predicted_id]
    plt.figure(figsize=(4, 4))
    plt.imshow(image_array)
    plt.title(f"Predicted sign: {predicted_label}")
    plt.axis("off")
    plt.show()

    print("Predicted sign:", predicted_label)
    print("Confidence:", f"{confidence:.2%}" if confidence is not None else
          "Not available as a calibrated probability for this model")
    return predicted_label

# Uncomment after setting a valid NEW_IMAGE_PATH:
# prediction = predict_sign(NEW_IMAGE_PATH)

## 23. Results summary and project conclusion

**Purpose:** Generate a concise, evidence-based conclusion from the actual held-out test metrics. This cell intentionally does not contain hard-coded scores.

When writing the final report, include:
- Number of classes and readable images.
- Train/validation/test image counts.
- Validation comparison of the five models.
- Selected model and why it was selected (validation macro F1).
- Final held-out test metrics and per-class classification report.
- Confusion-matrix observations and limitations.

Do not claim a model is highly accurate unless the displayed test results support that statement.

In [ ]:
print("========== INDIAN SIGN LANGUAGE RECOGNITION: FINAL SUMMARY ==========")
print("Total readable images:", len(image_df))
print("Number of sign classes:", len(label_names))
print("Train / Validation / Test:", len(train_df), "/", len(val_df), "/", len(test_df))
print("Selected model:", final_candidate_name)
print("\nHeld-out test results:")
for metric, value in test_metrics.items():
    print(f"{metric.replace('_', ' ').title():<20}: {value:.4f}")

macro_f1 = test_metrics["macro_f1"]
if macro_f1 >= 0.90:
    performance_note = "The model achieved strong macro-F1 on this held-out test split."
elif macro_f1 >= 0.75:
    performance_note = "The model achieved a useful baseline, but class-level errors should be reviewed."
else:
    performance_note = "The model needs further improvement; inspect class-level recall and the confusion matrix."

print("\nConclusion:", performance_note)
print("The selected model was chosen using validation data; the test set was reserved for final evaluation.")
print("Limitations: results depend on dataset size, image quality, signer diversity, backgrounds, and class balance.")

## 24. Suggested next improvements

1. Collect more images per sign, including different people, lighting, distances, and backgrounds.
2. Inspect classes with low recall and add targeted examples.
3. Compare a pretrained image model (for example, MobileNetV2) if the dataset and hardware support it.
4. Test on images captured from a real camera, not only images from the same dataset.
5. If the project needs continuous sign-language translation, note that this image-classification notebook recognizes isolated static signs; continuous signing requires video/temporal modelling and is a different task.

### Final checklist
- [ ] Dataset path points to the correct class-folder root.
- [ ] All classes and sample images were visually checked.
- [ ] Split sizes and class distributions were reviewed.
- [ ] All five models ran successfully.
- [ ] Validation comparison was generated.
- [ ] Final test metrics and confusion matrix were generated.
- [ ] Final model and metadata were saved.
- [ ] Conclusion was written using the actual output, not assumed results.